# 03 - CA-CWGAN with the k-NN filter (E5)

Trains CA-CWGAN (components E1-E5) and writes its synthetic pool (`N_MAX` rows
per class). The checkpoint it writes is reused by notebook 04. Each component
can be switched off for an ablation run. GPU. Run once per dataset (all
switches on). See `PIPELINE.md`, step 3.

In [ ]:
# Colab: upload df_train_<dataset>.csv (and ckpt_*.pt to resume an interrupted run).
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.mixture import BayesianGaussianMixture
from sklearn.neighbors import NearestNeighbors

try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATASET  = "diabetes"     # diabetes | yeast | ecoli | pageblocks
DATA_DIR = "."

SETTING = dict(updates=10000, batch=250, lr=2e-4, noise_dim=128)   # shared by every arm

SEED        = 42
CRITIC_ITER = 5           # critic steps per generator update
LAMBDA_GP   = 10          # E1: gradient-penalty coefficient
N_MODES     = 10          # E2: max mixture components per continuous column
TAU         = 0.0         # E3: q(y=k) ~ n_k^TAU, 0 = uniform over classes
GUMBEL_TAU  = 0.2         # E4a: Gumbel-softmax temperature
PAC         = 10          # E4b: samples per critic input (packing)
KNN_K       = 5           # E5: neighbours in the vote
KNN_MIN_OWN = 0.4         # E5: keep a row if >= 40% of its neighbours share its class
MAX_DROP    = 0.5         # E5: never discard more than half of a round
MAX_ROUNDS  = 8           # E5: generation rounds per class
MAX_DISCRETE_LEVELS = 12

# Speed only; these do not change what is generated or kept.
KNN_BACKEND = "auto"      # auto | torch | sklearn
KNN_CHUNK   = 1024
GEN_CHUNK   = 131072

MAX_MINUTES    = 150      # stop and checkpoint before a Colab session is cut
RESUME         = True
CHECKPOINT_DIR = "."      # a Google Drive path survives a disconnect
SAVE_EVERY     = 500

# Component switches. All True = CA-CWGAN; turn one off for an ablation run.
USE_E1_GRADIENT_PENALTY = True   # off: weight clipping at CLIP_VALUE
USE_E2_MODE_NORM        = True   # off: one scaled value per continuous column
USE_E3_BALANCED_COND    = True   # off: empirical class frequency
USE_E4_HEADS_AND_PACK   = True   # off: plain softmax heads, no packing
USE_E5_QUALITY_CONTROL  = True   # off: no post-generation filter
CLIP_VALUE = 0.01                # used only when E1 is off

# Active components, part of the file and checkpoint names ("12345" = full CA-CWGAN).
ACTIVE = "".join(t for t, on in [("1", USE_E1_GRADIENT_PENALTY), ("2", USE_E2_MODE_NORM),
                                 ("3", USE_E3_BALANCED_COND), ("4", USE_E4_HEADS_AND_PACK),
                                 ("5", USE_E5_QUALITY_CONTROL)] if on) or "none"

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. In Colab: Runtime > Change runtime type > T4 GPU.")
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{DATASET}.csv"))
TARGET_COL = df_train.columns[-1]
FEATURES = [c for c in df_train.columns if c != TARGET_COL]

_v = df_train[FEATURES].to_numpy(dtype=float)
assert _v.min() >= -1e-6 and _v.max() <= 1 + 1e-6, "features are not scaled to [0, 1]"

COUNTS = df_train[TARGET_COL].value_counts().sort_index()
N_MAX  = int(COUNTS.max())

DISCRETE_COLS, CONTINUOUS_COLS, LEVELS = [], [], {}
for col in FEATURES:
    lv = np.unique(df_train[col].to_numpy(dtype=float))
    if len(lv) <= MAX_DISCRETE_LEVELS:
        DISCRETE_COLS.append(col); LEVELS[col] = np.sort(lv)
    else:
        CONTINUOUS_COLS.append(col)

print("train:", df_train.shape, "| class counts:", COUNTS.tolist())
print("discrete columns:", len(DISCRETE_COLS), "| continuous:", len(CONTINUOUS_COLS))

## Class distribution (diagnostic plot, not in the paper)

In [ ]:
IR = float(COUNTS.max() / COUNTS.min())
fig, ax = plt.subplots(figsize=(7, 3.4))
bars = ax.bar([str(c) for c in COUNTS.index], COUNTS.to_numpy(), color="#4C72B0")
ax.bar_label(bars, fmt="%d", padding=3, fontsize=9)
ax.set_xlabel("class"); ax.set_ylabel("rows in df_train")
ax.set_title(f"{DATASET} - class distribution before augmentation (imbalance ratio {IR:.1f})")
ax.margins(y=0.16); ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

## E2 - Mode-specific normalisation

In [ ]:
class ColumnTransform:
    """Discrete -> one-hot. Continuous -> one-hot mode + offset (E2), or one scaled value."""

    def __init__(self, mode_norm=True):
        self.mode_norm = mode_norm

    def fit(self, df):
        self.spans = [(c, "discrete", len(LEVELS[c])) for c in DISCRETE_COLS]
        self.gmm, self.active = {}, {}
        for c in CONTINUOUS_COLS:
            if not self.mode_norm:
                self.spans.append((c, "plain", 1)); continue
            v = df[c].to_numpy(dtype=float).reshape(-1, 1)
            g = BayesianGaussianMixture(
                n_components=min(N_MODES, max(1, len(np.unique(v)))),
                weight_concentration_prior_type="dirichlet_process",
                weight_concentration_prior=1e-3, max_iter=100, random_state=SEED)
            g.fit(v)
            act = g.weights_ > 1e-3
            if not act.any():
                act[np.argmax(g.weights_)] = True
            self.gmm[c], self.active[c] = g, act
            self.spans.append((c, "vgm", 1 + int(act.sum())))
        self.dim = sum(w for _, _, w in self.spans)
        return self

    def transform(self, df):
        blocks = []
        for col, kind, width in self.spans:
            v = df[col].to_numpy(dtype=float)
            if kind == "discrete":
                lv = LEVELS[col]
                idx = np.abs(v[:, None] - lv[None, :]).argmin(axis=1)
                b = np.zeros((len(v), width), dtype=np.float32)
                b[np.arange(len(v)), idx] = 1.0
            elif kind == "plain":
                b = (v * 2 - 1).astype(np.float32).reshape(-1, 1)
            else:
                b = self._vgm(col, v, width)
            blocks.append(b)
        return np.concatenate(blocks, axis=1).astype(np.float32)

    def _vgm(self, col, v, width):
        g, act = self.gmm[col], self.active[col]
        mu = g.means_.reshape(-1)[act]
        sd = np.sqrt(g.covariances_.reshape(-1))[act]
        mode = g.predict_proba(v.reshape(-1, 1))[:, act].argmax(axis=1)
        alpha = np.clip((v - mu[mode]) / (4 * np.maximum(sd[mode], 1e-6)), -0.99, 0.99)
        b = np.zeros((len(v), width), dtype=np.float32)
        b[:, 0] = alpha
        b[np.arange(len(v)), 1 + mode] = 1.0
        return b

    def inverse(self, M):
        out, pos = {}, 0
        for col, kind, width in self.spans:
            blk = M[:, pos:pos + width]; pos += width
            if kind == "discrete":
                out[col] = LEVELS[col][blk.argmax(axis=1)]
            elif kind == "plain":
                out[col] = np.clip((blk[:, 0] + 1) / 2, 0, 1)
            else:
                g, act = self.gmm[col], self.active[col]
                mu = g.means_.reshape(-1)[act]
                sd = np.sqrt(g.covariances_.reshape(-1))[act]
                mode = blk[:, 1:].argmax(axis=1)
                out[col] = np.clip(np.clip(blk[:, 0], -1, 1) * 4 * sd[mode] + mu[mode], 0, 1)
        return pd.DataFrame(out)[FEATURES]

    def heads(self):
        """(activation, start, width) for every output block."""
        plan, pos = [], 0
        for col, kind, width in self.spans:
            if kind == "discrete":
                plan.append(("softmax", pos, width))
            elif kind == "plain":
                plan.append(("tanh", pos, 1))
            else:
                plan.append(("tanh", pos, 1)); plan.append(("softmax", pos + 1, width - 1))
            pos += width
        return plan

## Model: E1 gradient penalty, E4a mixed output heads, E4b packing

In [ ]:
PACK = PAC if USE_E4_HEADS_AND_PACK else 1


class Generator(nn.Module):
    def __init__(self, noise_dim, out_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim + cond_dim, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, out_dim),
        )

    def forward(self, z, y):
        return self.net(torch.cat([z, y], dim=1))


class Critic(nn.Module):
    def __init__(self, in_dim, cond_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear((in_dim + cond_dim) * PACK, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 256), nn.LeakyReLU(0.2), nn.Dropout(0.5),
            nn.Linear(256, 1),
        )

    def forward(self, x, y):
        j = torch.cat([x, y], dim=1)
        return self.net(j.reshape(-1, j.size(1) * PACK))


def activate(raw, hard=False):
    """tanh for mode offsets, Gumbel-softmax for modes and discrete levels."""
    parts = []
    for kind, start, width in CT.heads():
        blk = raw[:, start:start + width]
        if kind == "tanh":
            parts.append(torch.tanh(blk))
        elif USE_E4_HEADS_AND_PACK:
            parts.append(F.gumbel_softmax(blk, tau=GUMBEL_TAU, hard=hard, dim=1))
        else:
            parts.append(F.softmax(blk, dim=1))
    return torch.cat(parts, dim=1)


def gradient_penalty(D, real, fake, cond):
    a = torch.rand(real.size(0) // PACK, 1, 1, device=DEVICE)
    a = a.repeat(1, PACK, real.size(1)).reshape(-1, real.size(1))
    mixed = (a * real + (1 - a) * fake).requires_grad_(True)
    scores = D(mixed, cond)
    g = torch.autograd.grad(scores, mixed, torch.ones_like(scores),
                            create_graph=True, retain_graph=True, only_inputs=True)[0]
    return ((g.reshape(-1, PACK * real.size(1)).norm(2, dim=1) - 1) ** 2).mean() * LAMBDA_GP

## E3 - Class-balanced conditioning

In [ ]:
def prepare(df):
    """Fit the column transform and the conditioning distribution on df."""
    global CT                                  # read by activate()
    CT = ColumnTransform(mode_norm=USE_E2_MODE_NORM).fit(df)
    X_enc = CT.transform(df)
    labels = df[TARGET_COL].to_numpy()
    classes = np.sort(np.unique(labels))
    by_class = {int(k): np.flatnonzero(labels == k) for k in classes}
    counts = np.array([len(by_class[int(k)]) for k in classes], dtype=float)
    w = np.power(counts, TAU if USE_E3_BALANCED_COND else 1.0)
    return {"ct": CT, "X": torch.tensor(X_enc, device=DEVICE), "classes": classes,
            "cond_dim": len(classes), "by_class": by_class, "cond_p": w / w.sum(),
            "eye": torch.eye(len(classes), device=DEVICE)}

## Checkpointing

In [ ]:
def mount_if_drive(path):
    if str(path).startswith("/content/drive"):
        from google.colab import drive
        if not os.path.ismount("/content/drive"):
            drive.mount("/content/drive")
        os.makedirs(path, exist_ok=True)
    return path


CHECKPOINT_DIR = mount_if_drive(CHECKPOINT_DIR)


def trial_key(c):
    return f"u{c['updates']}_b{c['batch']}_lr{c['lr']:.0e}_z{c['noise_dim']}"


def ckpt_path(stem):
    return os.path.join(CHECKPOINT_DIR, f"ckpt_{stem}.pt")


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def rng_restore(s):
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([t.cpu() for t in s["cuda"]])


def save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim):
    """The RNG state saved at the end of training is what notebook 04 restores."""
    payload = {"version": 3, "kind": "ecwgan", "dataset": DATASET, "active": ACTIVE,
               "cfg": cfg, "updates_done": done, "history": history, "ct_dim": ct_dim,
               "G": G.state_dict(), "D": D.state_dict(),
               "opt_G": oG.state_dict(), "opt_D": oD.state_dict(),
               "rng": rng_state(), "seed": SEED}
    tmp = ckpt_path(stem) + ".tmp"
    torch.save(payload, tmp)
    os.replace(tmp, ckpt_path(stem))


def load_ckpt(stem, cfg, ct_dim):
    p = ckpt_path(stem)
    if not (RESUME and os.path.isfile(p)):
        return None
    c = torch.load(p, map_location=DEVICE, weights_only=False)
    if c.get("kind") != "ecwgan" or c.get("dataset") != DATASET or c.get("active") != ACTIVE:
        raise ValueError(f"{p} belongs to a different notebook or dataset")
    if trial_key(c["cfg"]) != trial_key(cfg) or c.get("ct_dim") != ct_dim:
        raise ValueError(f"{p} was trained with a different setting or training file")
    return c

## Training

In [ ]:
def train_ecwgan(ctx, cfg, stem, max_minutes, seed=SEED):
    """Train for cfg['updates'] generator updates (each with CRITIC_ITER critic steps).
    Stops and checkpoints when max_minutes runs out; call again to resume."""
    X_t = ctx["X"]
    eye, by_class = ctx["eye"], ctx["by_class"]
    classes, cond_dim, cond_p = ctx["classes"], ctx["cond_dim"], ctx["cond_p"]
    ct_dim = ctx["ct"].dim

    batch = max(PACK, (int(cfg["batch"]) // PACK) * PACK)
    log_every = max(1, cfg["updates"] // 200)

    G = Generator(cfg["noise_dim"], ct_dim, cond_dim).to(DEVICE)
    D = Critic(ct_dim, cond_dim).to(DEVICE)
    oG = torch.optim.Adam(G.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))
    oD = torch.optim.Adam(D.parameters(), lr=cfg["lr"], betas=(0.5, 0.9))

    def sample_cond(n):
        """Draw n conditions from cond_p and one real row of each drawn class."""
        drawn = np.random.choice(cond_dim, size=n, p=cond_p)
        rows = np.empty(n, dtype=np.int64)
        for ci in range(cond_dim):
            m = drawn == ci
            k = int(m.sum())
            if k:
                pool_idx = by_class[int(classes[ci])]
                rows[m] = pool_idx[np.random.randint(0, len(pool_idx), k)]
        return eye[torch.from_numpy(drawn).long().to(DEVICE)], rows

    ck = load_ckpt(stem, cfg, ct_dim)
    if ck is not None:
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        oG.load_state_dict(ck["opt_G"]); oD.load_state_dict(ck["opt_D"])
        rng_restore(ck["rng"])
        done, history = ck["updates_done"], list(ck["history"])
        print(f"resumed at {done:,} / {cfg['updates']:,} updates")
    else:
        set_seed(seed)
        done, history = 0, []
    if done >= cfg["updates"]:
        return G, history, done

    deadline = time.time() + max_minutes * 60
    t_start, started_at = time.time(), done
    for u in range(done, cfg["updates"]):
        dl = 0.0
        for _ in range(CRITIC_ITER):
            cond, rows = sample_cond(batch)
            real = X_t[torch.from_numpy(rows).long().to(DEVICE)]
            z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
            fake = activate(G(z, cond))
            loss_D = D(fake.detach(), cond).mean() - D(real, cond).mean()
            if USE_E1_GRADIENT_PENALTY:
                loss_D = loss_D + gradient_penalty(D, real, fake.detach(), cond)
            oD.zero_grad(set_to_none=True); loss_D.backward(); oD.step()
            if not USE_E1_GRADIENT_PENALTY:              # fall back to weight clipping
                with torch.no_grad():
                    for p in D.parameters():
                        p.clamp_(-CLIP_VALUE, CLIP_VALUE)
            dl += float(loss_D.detach())

        cond, _ = sample_cond(batch)
        z = torch.randn(batch, cfg["noise_dim"], device=DEVICE)
        loss_G = -D(activate(G(z, cond)), cond).mean()
        oG.zero_grad(set_to_none=True); loss_G.backward(); oG.step()

        done = u + 1
        if done % log_every == 0 or done == cfg["updates"]:
            history.append((done, dl / CRITIC_ITER, float(loss_G.detach())))
        if done % SAVE_EVERY == 0:
            save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim)
        if done % (log_every * 10) == 0:
            rate = (done - started_at) / max(time.time() - t_start, 1e-9)
            print(f"  update {done:>7,} / {cfg['updates']:,} | D {dl / CRITIC_ITER:9.4f} "
                  f"| G {float(loss_G):9.4f} | {rate:6.1f} upd/s")
        if time.time() > deadline:
            print(f"time limit at {done:,} updates - re-run to resume from the checkpoint")
            break

    save_ckpt(stem, cfg, G, D, oG, oD, done, history, ct_dim)
    return G, history, done

## E5 - Per-class alignment and k-NN vote

In [ ]:
def knn_indices(reference, query, k):
    """Indices of the k nearest reference rows for every query row.
    The torch path is the same computation on the GPU, used for large problems."""
    big = len(query) * len(reference) > 2e8
    use_torch = KNN_BACKEND == "torch" or (
        KNN_BACKEND == "auto" and torch.cuda.is_available() and big)
    if not use_torch:
        nn_ = NearestNeighbors(n_neighbors=k, n_jobs=-1).fit(reference)
        return nn_.kneighbors(query, return_distance=False)

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    R = torch.as_tensor(np.ascontiguousarray(reference), dtype=torch.float32, device=dev)
    out = np.empty((len(query), k), dtype=np.int64)
    s, chunk = 0, KNN_CHUNK
    while s < len(query):
        try:
            Q = torch.as_tensor(np.ascontiguousarray(query[s:s + chunk]),
                                dtype=torch.float32, device=dev)
            out[s:s + chunk] = torch.cdist(Q, R).topk(k, dim=1, largest=False).indices.cpu().numpy()
            s += chunk
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if chunk <= 64:
                raise
            chunk //= 2
    del R
    if dev == "cuda":
        torch.cuda.empty_cache()
    return out


def make_generate(G, ctx, cfg):
    """A function returning n decoded rows of one class."""
    classes, eye, ct = ctx["classes"], ctx["eye"], ctx["ct"]
    step = max(PACK, (GEN_CHUNK // PACK) * PACK)

    def generate(cls, n):
        idx = int(np.flatnonzero(classes == cls)[0])
        chunks, made = [], 0
        with torch.no_grad():
            while made < n:
                size = max(min(step, n - made), PACK)
                z = torch.randn(size, cfg["noise_dim"], device=DEVICE)
                cond = eye[torch.full((size,), idx, dtype=torch.long, device=DEVICE)]
                chunks.append(activate(G(z, cond), hard=True).cpu().numpy())
                made += size
        return ct.inverse(np.concatenate(chunks)[:n])

    return generate


def make_filters(ref_df):
    """align() and denoise(), both measured against the real training rows."""
    ref_X = ref_df[FEATURES].to_numpy(dtype=np.float32)
    ref_y = ref_df[TARGET_COL].to_numpy()
    ranges = {c: (g[FEATURES].min(), g[FEATURES].max()) for c, g in ref_df.groupby(TARGET_COL)}

    def align(df_part, cls):
        """Clip to the range of that class; snap discrete columns to legal levels."""
        out = df_part.copy()
        lo, hi = ranges[cls]
        out[FEATURES] = out[FEATURES].clip(lo, hi, axis=1)
        for col in FEATURES:
            if col in LEVELS:
                lv = LEVELS[col]
                v = out[col].to_numpy(dtype=float)
                pos = np.clip(np.searchsorted(lv, v), 1, len(lv) - 1)
                left, right = lv[pos - 1], lv[pos]
                out[col] = np.where(np.abs(v - left) <= np.abs(v - right), left, right)
        return out

    def denoise(df_part, cls):
        """Keep rows whose real neighbours mostly share their label, capped at MAX_DROP."""
        k = min(KNN_K, len(ref_df) - 1)
        idx = knn_indices(ref_X, df_part[FEATURES].to_numpy(dtype=np.float32), k)
        own = (ref_y[idx] == cls).mean(axis=1)
        keep = own >= KNN_MIN_OWN
        cap = int(len(df_part) * MAX_DROP)
        if (~keep).sum() > cap:
            order = np.argsort(-own)
            keep = np.zeros(len(df_part), bool)
            keep[order[:len(df_part) - cap]] = True
        return df_part.loc[keep].reset_index(drop=True)

    return align, denoise


def build_pool(G, ctx, cfg, ref_df, per_class):
    """Generate, align and filter until every class holds per_class rows."""
    generate = make_generate(G, ctx, cfg)
    align, denoise = make_filters(ref_df)

    G.eval()
    parts, log = [], []
    for cls in ctx["classes"]:
        kept, have, drawn, rounds = [], 0, 0, 0
        while have < per_class and rounds < MAX_ROUNDS:
            short = per_class - have
            want = int(np.ceil(short / (1 - MAX_DROP))) if USE_E5_QUALITY_CONTROL else short
            part = generate(cls, want)
            part[TARGET_COL] = cls
            drawn += want
            if USE_E5_QUALITY_CONTROL:
                part = denoise(align(part, cls), cls)
            kept.append(part)
            have += len(part)
            rounds += 1
        part = pd.concat(kept, ignore_index=True).iloc[:per_class].reset_index(drop=True)
        if len(part) < per_class:
            raise RuntimeError(f"class {cls}: only {len(part):,} of {per_class:,} rows survived")
        parts.append(part)
        log.append({"class": cls, "kept": len(part), "generated": drawn, "rounds": rounds})
    G.train()
    display(pd.DataFrame(log))
    return pd.concat(parts, ignore_index=True)[ref_df.columns]

## Run

In [ ]:
STEM = f"ecwgan_E{ACTIVE}_{DATASET}_{trial_key(SETTING)}_seed{SEED}"
CTX = prepare(df_train)
print("conditioning share per class:", np.round(CTX["cond_p"], 3).tolist())
steps_per_epoch = max(1, len(df_train) // SETTING["batch"])
print("setting   :", trial_key(SETTING))
print("equivalent:", f"{SETTING['updates'] / steps_per_epoch:.1f} epochs on this dataset")

G, history, done = train_ecwgan(CTX, SETTING, STEM, MAX_MINUTES)
COMPLETE = done >= SETTING["updates"]
print("training complete:", COMPLETE)

In [ ]:
if COMPLETE:
    df_pool = build_pool(G, CTX, SETTING, df_train, N_MAX)
    valid = [np.isin(df_pool[c].to_numpy(dtype=float), LEVELS[c]).mean() for c in DISCRETE_COLS]
    print(f"valid discrete cells: {100 * np.mean(valid) if valid else 100:.2f}%")

    assert list(df_pool.columns) == list(df_train.columns)
    assert df_pool.isna().sum().sum() == 0
    assert (df_pool[TARGET_COL].value_counts() == N_MAX).all()

    POOL_FILE = os.path.join(DATA_DIR, f"{STEM}_pool.csv")
    df_pool.to_csv(POOL_FILE, index=False)
    print("written:", POOL_FILE, f"({len(df_pool):,} rows)")
    if IN_COLAB:
        files.download(POOL_FILE)

## Composition of the augmented training set (diagnostic plot, not in the paper)

In [ ]:
if COMPLETE:
    real = COUNTS.astype(int)
    added = (N_MAX - real).clip(lower=0)
    top = N_MAX
    fig, ax = plt.subplots(figsize=(max(7.5, 0.95 * len(real) + 3), 4.4))
    x = [str(c) for c in real.index]
    ax.bar(x, real, label="real (df_train)", color="#4C72B0")
    ax.bar(x, added, bottom=real, label="synthetic added", color="#DD8452")
    floor = top * 0.07
    for i, (r, a) in enumerate(zip(real, added)):
        if r >= floor:
            ax.text(i, r / 2, f"{r:,}", ha="center", va="center", fontsize=8, color="white")
        if a >= floor:
            ax.text(i, r + a / 2, f"+{a:,}", ha="center", va="center", fontsize=8, color="white")
        ax.text(i, r + a + top * 0.015, f"{r + a:,}", ha="center", va="bottom", fontsize=8)
    ax.set_ylim(0, top * 1.18)
    ax.set_xlabel("class"); ax.set_ylabel("rows")
    ax.set_title(f"{DATASET} - composition of the augmented training set", pad=12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0, fontsize=9)
    ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout(); plt.show()

## Real against synthetic (diagnostic plot, not in the paper)

The same number of rows (the smallest class size) is drawn from every class of
both sets, so the plots compare columns rather than class mix.

In [ ]:
if COMPLETE:
    SAMPLE_N = int(COUNTS.min())

    def balanced_sample(df, n, seed=SEED):
        return pd.concat([g.sample(min(n, len(g)), random_state=seed)
                          for _, g in df.groupby(TARGET_COL)]).reset_index(drop=True)

    real_s = balanced_sample(df_train, SAMPLE_N)
    synth_s = balanced_sample(df_pool, SAMPLE_N)

    ncol = 4
    nrow = int(np.ceil(len(FEATURES) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 2.4 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, FEATURES):
        both = np.concatenate([real_s[col].to_numpy(dtype=float), synth_s[col].to_numpy(dtype=float)])
        bins = np.histogram_bin_edges(both, bins=min(20, max(3, len(np.unique(both)))))
        ax.hist(real_s[col], bins=bins, density=True, alpha=0.55, color="#4C72B0", label="real")
        ax.hist(synth_s[col], bins=bins, density=True, alpha=0.55, color="#DD8452", label="synthetic")
        ax.set_title(col, fontsize=9); ax.tick_params(labelsize=7); ax.set_yticks([])
    for ax in axes[len(FEATURES):]:
        ax.axis("off")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{DATASET} - per-column distribution, real vs synthetic "
                 f"({SAMPLE_N:,} rows per class from each)", y=1.002)
    plt.tight_layout(); plt.show()

    pos = np.arange(len(FEATURES))
    fig, ax = plt.subplots(figsize=(max(9, 0.62 * len(FEATURES)), 4.8))
    b1 = ax.boxplot([real_s[c].to_numpy(dtype=float) for c in FEATURES], positions=pos - 0.19,
                    widths=0.32, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black"))
    b2 = ax.boxplot([synth_s[c].to_numpy(dtype=float) for c in FEATURES], positions=pos + 0.19,
                    widths=0.32, patch_artist=True, showfliers=False,
                    medianprops=dict(color="black"))
    for box in b1["boxes"]:
        box.set_facecolor("#4C72B0"); box.set_alpha(0.75)
    for box in b2["boxes"]:
        box.set_facecolor("#DD8452"); box.set_alpha(0.75)
    ax.set_xticks(pos); ax.set_xticklabels(FEATURES, rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("value (columns are Min-Max scaled to [0, 1])")
    ax.set_title(f"{DATASET} - spread per column, real vs synthetic "
                 f"({SAMPLE_N:,} rows per class from each). Outliers hidden.")
    ax.legend([b1["boxes"][0], b2["boxes"][0]], ["real", "synthetic"], loc="upper right")
    ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    plt.tight_layout(); plt.show()

In [ ]:
# Keep the checkpoint: notebook 04 re-samples this generator from it.
if IN_COLAB and not str(CHECKPOINT_DIR).startswith("/content/drive"):
    files.download(ckpt_path(STEM))